# Restaurant Survival Classification — Final Model

**Algorithm:** Elastic-net Logistic Regression + Bagged Decision Trees (probability blend, weights 0.775 / 0.225)  
**Selection criterion:** Balanced Accuracy (BA) maximised on a stratified 20% hold-out validation set  
**Decision threshold:** tuned on hold-out validation probabilities to maximise BA

**Pipeline:**
1. Load training & test data
2. Feature engineering (log transforms, recency ratios, local competition metrics)
3. 80/20 stratified train/validation split + preprocessing
4. Fit elastic-net logistic and bagging ensemble; blend probabilities; tune threshold on validation set
5. Refit both models on full training data → predict on test → save `submission_latest.csv`


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

RNG = 42
pd.set_option('display.max_columns', 120)


In [2]:
TRAIN_PATH = 'restaurants_train.csv'
TEST_PATH  = 'restaurants_test.csv'
ID, TARGET = 'restaurant_id', 'status_closed'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# Cast boolean columns to nullable int so sklearn handles them numerically
for df in (train, test):
    for c in df.select_dtypes(include='bool').columns:
        df[c] = df[c].astype('Int64')

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Positive (closed) rate: {train[TARGET].mean():.2%}")


Train: (33296, 86)  |  Test: (8325, 85)
Positive (closed) rate: 9.81%


In [3]:
HIGH_NA_COLS = [
    'tagcat_payment_options', 'price_level', 'tagcat_social_inclusivity',
    'ratings_avg_12m_prior', 'ratings_num_12m_prior'
]

LOG_COLS = [
    'user_ratings_total',
    'ratings_num_1m_prior', 'ratings_num_3m_prior', 'ratings_num_6m_prior',
    'ratings_num_9m_prior', 'ratings_num_12m_prior',
    'lang_pl_count', 'rating_pl', 'rating_foreign',
    'review_length_avg', 'review_length_std',
    'catch_restaurant_count_500m', 'catch_restaurant_count_1000m', 'catch_restaurant_count_2000m',
    'residents', 'restaurant_count',
    'poi_count_100m', 'poi_count_200m', 'poi_count_500m', 'poi_count_1000m', 'poi_count_2000m',
    'place_age_days'
]


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    """Apply feature engineering transformations to raw data."""
    df = df.copy()

    # Missing indicators for high-missingness columns
    for col in HIGH_NA_COLS:
        if col in df.columns:
            df[f'{col}__isna'] = df[col].isna().astype(int)

    df['missing_count'] = df.isna().sum(axis=1)

    # Log transforms on heavy-tailed count columns
    for col in LOG_COLS:
        if col in df.columns:
            df[f'{col}__log'] = np.log1p(df[col].clip(lower=0))

    # Recency share of total reviews
    if 'user_ratings_total' in df.columns:
        total = df['user_ratings_total'].clip(lower=0).replace(0, np.nan)
        for src, new_col in [
            ('ratings_num_1m_prior',  'recent_1m_share'),
            ('ratings_num_3m_prior',  'recent_3m_share'),
            ('ratings_num_6m_prior',  'recent_6m_share'),
            ('ratings_num_12m_prior', 'recent_12m_share'),
        ]:
            if src in df.columns:
                df[new_col] = df[src].clip(lower=0) / total

    # Momentum / burst features
    if {'ratings_num_1m_prior', 'ratings_num_3m_prior'}.issubset(df.columns):
        r1 = df['ratings_num_1m_prior'].clip(lower=0)
        r3 = df['ratings_num_3m_prior'].clip(lower=0)
        df['momentum_1m_to_3m']      = (r1 + 1) / (r3 + 1)
        df['recent_burst_1m_vs_3m']  = r1 - r3 / 3.0

    if {'ratings_num_3m_prior', 'ratings_num_12m_prior'}.issubset(df.columns):
        r3  = df['ratings_num_3m_prior'].clip(lower=0)
        r12 = df['ratings_num_12m_prior'].clip(lower=0)
        df['momentum_3m_to_12m']      = (r3 + 1) / (r12 + 1)
        df['recent_burst_3m_vs_12m']  = r3 - r12 / 4.0

    # Hours / weekend share
    if {'hours_open', 'hours_open_weekends'}.issubset(df.columns):
        total_hours = df['hours_open'].replace(0, np.nan)
        df['weekend_hours_share'] = df['hours_open_weekends'] / total_hours
        df['weekday_hours']       = df['hours_open'] - df['hours_open_weekends']

    # Ratings velocity
    if {'user_ratings_total', 'place_age_days'}.issubset(df.columns):
        age   = df['place_age_days'].clip(lower=0)
        total = df['user_ratings_total'].clip(lower=0)
        df['ratings_per_day'] = total / (age + 1)
        if 'ratings_num_12m_prior' in df.columns:
            df['recent_year_per_day'] = df['ratings_num_12m_prior'].clip(lower=0) / np.minimum(age + 1, 365)

    # Local competition metrics (retained — shown to help BA)
    if {'catch_restaurant_count_500m', 'user_ratings_total'}.issubset(df.columns):
        local_comp = df['catch_restaurant_count_500m'].clip(lower=0)
        total      = df['user_ratings_total'].clip(lower=0)
        df['reviews_per_local_restaurant'] = total / (local_comp + 1)
        df['popularity_vs_competition']    = np.log1p(total) / (local_comp + 1)

    if {'restaurant_count', 'residents'}.issubset(df.columns):
        residents = df['residents'].clip(lower=0)
        df['restaurant_density'] = df['restaurant_count'].clip(lower=0) / (residents + 1)

    if {'poi_count_500m', 'residents'}.issubset(df.columns):
        residents = df['residents'].clip(lower=0)
        df['poi_density_500m'] = df['poi_count_500m'].clip(lower=0) / (residents + 1)

    # Rating strength interactions
    if {'rating_avg', 'user_ratings_total'}.issubset(df.columns):
        total = df['user_ratings_total'].clip(lower=0)
        df['rating_x_popularity'] = df['rating_avg'] * np.log1p(total)

    if {'rating_avg', 'place_age_days'}.issubset(df.columns):
        age = df['place_age_days'].clip(lower=0)
        df['rating_x_age'] = df['rating_avg'] * np.log1p(age)

    if {'rating_avg', 'rating_std'}.issubset(df.columns):
        df['rating_consistency'] = df['rating_avg'] / (df['rating_std'].clip(lower=0) + 1)

    # Engagement features
    if {'review_has_text_pct', 'review_length_avg'}.issubset(df.columns):
        text_pct = df['review_has_text_pct'].clip(lower=0)
        length   = df['review_length_avg'].clip(lower=0)
        df['engagement_score'] = text_pct * np.log1p(length)

    if {'review_has_text_pct', 'user_ratings_total'}.issubset(df.columns):
        text_pct = df['review_has_text_pct'].clip(lower=0)
        total    = df['user_ratings_total'].clip(lower=0)
        df['text_review_volume'] = text_pct * np.log1p(total)

    # Tag count signal
    tag_cols = [col for col in df.columns if col.startswith('tagcat_') and col != 'tagcat_payment_options']
    if tag_cols:
        df['tag_signal_count'] = df[tag_cols].fillna(0).sum(axis=1)

    # Drop noisy / low-signal columns
    df = df.drop(columns=['tagcat_payment_options'], errors='ignore')
    df = df.drop(columns=[col for col in df.columns if col.startswith('gus_')], errors='ignore')

    return df


train_fe = engineer(train)
test_fe  = engineer(test)
print(f"train_fe: {train_fe.shape}  |  test_fe: {test_fe.shape}")
added = sorted(set(train_fe.columns) - set(train.columns))
print(f"Added {len(added)} engineered features")


train_fe: (33296, 122)  |  test_fe: (8325, 121)
Added 50 engineered features


In [4]:
y = train_fe[TARGET].astype(int)
X = train_fe.drop(columns=[TARGET, ID])
X_test   = test_fe.drop(columns=[ID])
test_ids = test_fe[ID]

# 80/20 stratified train/validation split — threshold is tuned on the validation set
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RNG)
print(f"train: {X_tr.shape}  val: {X_val.shape}")

CAT_COLS = ['category_top20']
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]

# Scaled preprocessor for elastic-net logistic regression
pre_scaled = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='mean')),
        ('sc',  StandardScaler()),
    ]), NUM_COLS),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh',  OneHotEncoder(handle_unknown='ignore')),
    ]), CAT_COLS),
])

# Unscaled preprocessor for tree-based bagging
pre_tree = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='mean')),
    ]), NUM_COLS),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh',  OneHotEncoder(handle_unknown='ignore')),
    ]), CAT_COLS),
])

print(f"Features: {len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical")


train: (26636, 120)  val: (6660, 120)
Features: 119 numeric + 1 categorical


In [5]:
# ─── Chosen algorithm: Elastic-net Logistic Regression + Bagging blend ────────
# Justification: probability blending of a regularised linear model and a
# tree-based ensemble captures both linear separation and non-linear structure.
# Elastic-net handles class imbalance via explicit class weighting and performs
# embedded feature selection (l1_ratio=0.10 ≈ ridge with sparse correction).
# Bagging adds low-variance non-linear predictions on competition/recency features.
# Blend weights (0.775 enet / 0.225 bag) selected by ensemble comparison in
# restaurant_survival_exploration.ipynb.
# Decision threshold tuned on the 20% hold-out validation set.
# ─────────────────────────────────────────────────────────────────────────────

pos_ratio  = (y_tr == 0).sum() / (y_tr == 1).sum()
BLEND_ENET = 0.775
BLEND_BAG  = 0.225

enet = Pipeline([
    ('pre', pre_scaled),
    ('clf', LogisticRegression(
        max_iter=6000, penalty='elasticnet', solver='saga',
        C=0.20, l1_ratio=0.10,
        class_weight={0: 1.0, 1: pos_ratio * 0.80},
        random_state=RNG,
    )),
])

bag = Pipeline([
    ('pre', pre_tree),
    ('clf', BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_leaf=20, class_weight='balanced', random_state=RNG),
        n_estimators=400, max_samples=0.85, bootstrap=True,
        n_jobs=-1, random_state=RNG,
    )),
])

print("Fitting elastic-net logistic regression ...")
enet.fit(X_tr, y_tr)
print("Fitting bagged decision trees ...")
bag.fit(X_tr, y_tr)

# Blend and tune threshold on the hold-out validation set
def blend_proba(X_new):
    return BLEND_ENET * enet.predict_proba(X_new)[:, 1] + BLEND_BAG * bag.predict_proba(X_new)[:, 1]

p_val    = blend_proba(X_val)
thr_grid = np.linspace(0.30, 0.65, 351)
bas      = [balanced_accuracy_score(y_val, (p_val >= t).astype(int)) for t in thr_grid]
t_star   = float(thr_grid[int(np.argmax(bas))])
ba_star  = float(np.max(bas))
ba_05    = balanced_accuracy_score(y_val, (p_val >= 0.5).astype(int))

val_pred = (p_val >= t_star).astype(int)
print(f"\nHold-out BA@0.5 : {ba_05:.4f}")
print(f"Hold-out BA*    : {ba_star:.4f}  (threshold = {t_star:.3f})")
print("\nValidation confusion matrix:")
print(confusion_matrix(y_val, val_pred))
print(classification_report(y_val, val_pred, digits=3))


Fitting elastic-net logistic regression ...
Fitting bagged decision trees ...

Hold-out BA@0.5 : 0.6711
Hold-out BA*    : 0.6904  (threshold = 0.431)

Validation confusion matrix:
[[4179 1827]
 [ 206  448]]
              precision    recall  f1-score   support

           0      0.953     0.696     0.804      6006
           1      0.197     0.685     0.306       654

    accuracy                          0.695      6660
   macro avg      0.575     0.690     0.555      6660
weighted avg      0.879     0.695     0.755      6660



In [7]:
# Refit BOTH models on the FULL training set then apply to test
print("Refitting both models on all training data ...")
pos_ratio_full = (y == 0).sum() / (y == 1).sum()

enet_full = Pipeline([
    ('pre', pre_scaled),
    ('clf', LogisticRegression(
        max_iter=6000, penalty='elasticnet', solver='saga',
        C=0.20, l1_ratio=0.10,
        class_weight={0: 1.0, 1: pos_ratio_full * 0.80},
        random_state=RNG,
    )),
])
bag_full = Pipeline([
    ('pre', pre_tree),
    ('clf', BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_leaf=20, class_weight='balanced', random_state=RNG),
        n_estimators=400, max_samples=0.85, bootstrap=True,
        n_jobs=-1, random_state=RNG,
    )),
])

enet_full.fit(X, y)
bag_full.fit(X, y)

# Blend test probabilities and apply the threshold tuned on validation
p_test      = BLEND_ENET * enet_full.predict_proba(X_test)[:, 1] + BLEND_BAG * bag_full.predict_proba(X_test)[:, 1]
y_test_pred = (p_test >= t_star).astype(int)

submission = pd.DataFrame({ID: test_ids.values, TARGET: y_test_pred})
submission.to_csv('submission_latest.csv', index=False)

print(f"Saved: submission_latest.csv  —  shape: {submission.shape}")
print(f"Predicted closure rate on test : {y_test_pred.mean():.3f}  "
      f"(train baseline: {y.mean():.3f})")
print(f"Expected test BA (hold-out val): {ba_star:.4f}  (threshold: {t_star:.3f})")
submission.head()


Refitting both models on all training data ...
Saved: submission_latest.csv  —  shape: (8325, 2)
Predicted closure rate on test : 0.322  (train baseline: 0.098)
Expected test BA (hold-out val): 0.6904  (threshold: 0.431)


,restaurant_id,status_closed
0,192311983,0
1,621181627,0
2,119760055,0
3,729596929,0
4,116589396,1
